## Neccessary Imports

In [1]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_classic.chains import RetrievalQA

## Loading Data From PDFs

In [ ]:
# loading all pdfs in a folder
def load_directory_pdfs(directory_path):
    # this looks for all .pdf files in the folder
    loader = DirectoryLoader(directory_path, glob="./*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

# i have stored all the pdfs in a folder called "data" in the current directory
pdf_docs = load_directory_pdfs("data")

print(f"Loaded {len(pdf_docs)} pages/documents.")

Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 132 0 (offset 0)
Ignoring wrong pointing object 134 0 (offset 0)
Ignoring wrong pointing object 152 0 (offset 0)
Ignoring wrong pointing object 154 0 (offset 0)
Ignoring wrong pointing object 164 0 (offset 0)
Ignoring wrong pointing object 186 0 (offset 0)
Ignoring wrong pointing object 197 0 (offset 0)
Ignor

Loaded 814 pages/documents.


## Splitting Documents into Chunks

In [ ]:
# splitting docs into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum of 500 characters per chunk
    chunk_overlap=100,  # Overlapping 100 characters to preserve context
    length_function=len,  # Determines chunk size based on character length
    is_separator_regex=False,  # The separator used for splitting is not a regex pattern
)

# splitting documents into smaller chunks
split_documents = text_splitter.split_documents(pdf_docs)

## Embedding Documents and Storing in Chroma

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. initializing Embeddings 
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# 2. setting directory
persist_directory = "./chroma_db"

# 3. Storing documents (This SAVES them automatically)
vectorstore = Chroma.from_documents(
    documents=split_documents, 
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Data successfully stored in ChromaDB!")

# 4. Reloading to verify it works
vectorstore = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embeddings
)
print(f"ChromaDB reloaded! There are {vectorstore._collection.count()} chunks in the database.")

Data successfully stored in ChromaDB!
ChromaDB reloaded! There are 2882 chunks in the database.


## Retrieving Relevant Documents

In [ ]:
# RECONNECTING TO CHROMADB LATER ON (e.g. in a new session or notebook)
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. Initializing the same embedding model we used before
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# 2. Pointinng to the folder where data lives
persist_directory = "./chroma_db"

# 3. Re-creating the 'vectorstore' variable by loading it from disk
vectorstore = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embeddings
)

print(f"Reconnected! Ready to search through {vectorstore._collection.count()} chunks.")

Reconnected! Ready to search through 2882 chunks.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

# 1. initializing the LLM 
llm = ChatOllama(model="tinyllama")

# 2. Setting up the "Search Engine" (The Retriever), 
# this looks at the 2,882 chunks and finds the best matches
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. creating the Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# 4. asking a question to the chain, which will search through the chunks and use the LLM to answer
query = "what is the purpose of OS?" 
response = qa_chain.invoke(query)

print("\n--- ANSWER ---")
print(response["result"])


--- ANSWER ---
The purpose of operating systems (OS) is to provide a way for the hardware to be used by applications and users. The OS provides abstracted interfaces that allow each application to use the hardware as if it were just another device in the computer system. By providing these interfaces, operating systems enable the creation of advanced software and hardware products.

In addition to this, the OS helps in managing the system's resources such as memory, processing power, storage, and peripherals. It also provides security features to protect the user from unauthorized access or exploitation of the system's resources. All these features help in providing better and faster functionality for users while keeping the hardware operational and stable.

In simpler words, the OS is a "bridge" between applications and the hardware. It allows one process to use the hardware resources efficiently and provide smooth user experience.
